In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

data = np.load("20000.npy")
df = pd.DataFrame(data)
df["palya_id"] = df["eventID"].astype(str) + "_" + df["trackID"].astype(str)
df = df[df["parentID"] == 0].copy()
parameters = ['posX', 'posY', 'edep'] 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
epochs = 300

def prepare_data(layerA, layerB, params):
    common_labels = np.array(list(set(layerA["palya_id"]) & set(layerB["palya_id"])))
    np.random.shuffle(common_labels)
    train_split = int(0.9 * len(common_labels))
    
    dfA_subset = layerA.set_index("palya_id").loc[common_labels[:train_split]]
    mins = torch.tensor(dfA_subset[params].min().values).to(device)
    maxs = torch.tensor(dfA_subset[params].max().values).to(device)
    
    def scale(df_subset):
        return (torch.tensor(df_subset[params].values).to(device) - mins) / (maxs - mins + 1e-6)

    l_train = [scale(layerA.set_index("palya_id").loc[common_labels[:train_split]]), 
               scale(layerB.set_index("palya_id").loc[common_labels[:train_split]])]
    l_val = [scale(layerA.set_index("palya_id").loc[common_labels[train_split:]]), 
             scale(layerB.set_index("palya_id").loc[common_labels[train_split:]])]
    return l_train, l_val, mins, maxs

# 3. Modell architektúra
class Matcher(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim) 
        )

    def forward(self, x):
        return self.net(x)

# 4. Segédfüggvény: Pontosság mérése
def calculate_accuracy_mm(pred, target, mins, maxs, threshold_mm=2.0):
    scale_factor = (maxs - mins)
    pred_real = pred * scale_factor
    target_real = target * scale_factor
    dist_mm = torch.norm(pred_real[:, :2] - target_real[:, :2], dim=1)
    
    return (dist_mm < threshold_mm).float().mean().item()

# 5. Tanítási folyamat
def train(l_train, l_val, mins, maxs, name):
    print(f"\n Tanítás: {name} ")
    model = Matcher(len(parameters)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()
    
    for e in range(epochs):
        model.train()
        indices = torch.randperm(len(l_train[0]))
        train_accs = []
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i+batch_size]
            x_prev, x_next = l_train[0][batch_idx], l_train[1][batch_idx]
            
            optimizer.zero_grad()
            pred = model(x_prev)
            
            loss = criterion(pred, x_next) 
            loss.backward()
            optimizer.step()
            train_accs.append(calculate_accuracy_mm(pred, x_next, mins, maxs))
        
        model.eval()
        with torch.no_grad():
            v_acc = calculate_accuracy_mm(model(l_val[0]), l_val[1], mins, maxs)
        
        if e % 20 == 0:
            print(f"Epoch {e:3d} | Train Acc: {np.mean(train_accs)*100:6.2f}% | Val Acc: {v_acc*100:6.2f}%")
    return model

pairs = [([15, 16], "L15-L16"), ([16, 17], "L16-L17")]
models = {}

for indices, name in pairs:
    lA = df[df["volumeID[3]"] == indices[0]].drop_duplicates(subset=["palya_id"])
    lB = df[df["volumeID[3]"] == indices[1]].drop_duplicates(subset=["palya_id"])
    
    train_data, val_data, mins, maxs = prepare_data(lA, lB, parameters)
    models[name] = train(train_data, val_data, mins, maxs, name)


 Tanítás: L15-L16 
Epoch   0 | Train Acc:   1.61% | Val Acc:   2.80%
Epoch  20 | Train Acc:  96.41% | Val Acc:  97.20%
Epoch  40 | Train Acc:  95.53% | Val Acc:  96.37%
Epoch  60 | Train Acc:  95.24% | Val Acc:  95.81%
Epoch  80 | Train Acc:  95.38% | Val Acc:  95.32%
Epoch 100 | Train Acc:  95.19% | Val Acc:  95.81%
Epoch 120 | Train Acc:  95.19% | Val Acc:  96.09%
Epoch 140 | Train Acc:  95.21% | Val Acc:  93.99%
Epoch 160 | Train Acc:  95.17% | Val Acc:  95.95%
Epoch 180 | Train Acc:  95.17% | Val Acc:  96.44%
Epoch 200 | Train Acc:  95.12% | Val Acc:  95.60%
Epoch 220 | Train Acc:  95.15% | Val Acc:  95.39%
Epoch 240 | Train Acc:  95.04% | Val Acc:  95.67%
Epoch 260 | Train Acc:  95.06% | Val Acc:  95.60%
Epoch 280 | Train Acc:  95.33% | Val Acc:  95.39%

 Tanítás: L16-L17 
Epoch   0 | Train Acc:   1.00% | Val Acc:   2.97%
Epoch  20 | Train Acc:  95.45% | Val Acc:  95.83%
Epoch  40 | Train Acc:  95.34% | Val Acc:  95.76%
Epoch  60 | Train Acc:  95.30% | Val Acc:  95.05%
Epoch  80 